# SE-BRL Data Preprocessing

## Purpose

This notebook prepares the cleaned SE-BRL datasets for leakage-aware feature
extraction and machine-learning experiments.

It builds on the outputs of `02_data_cleaning.ipynb` and converts the cleaned
source datasets into consistent representations while preserving modality,
source provenance, native labels, and grouping information required by the
experimental design.

## Preprocessing Principles

The following principles are applied throughout this notebook:

- cleaned datasets under `experiments/data/interim/` are treated as inputs;
- source-native labels and dataset provenance are preserved;
- target-label mappings must be supported by documented source semantics;
- spam is not automatically treated as equivalent to phishing;
- duplicate or related observations must not cross experimental partitions;
- grouping is established before partition assignment;
- development, calibration, and test partitions follow the predefined
  70/15/15 experimental design;
- the sealed test partition is not used for model development or threshold
  selection;
- learned preprocessing parameters are fitted only on appropriate training
  data;
- cross-validation preprocessing is fitted independently within each
  development fold; and
- preprocessing must not introduce human-assigned behavioral annotations.

## Inputs

Cleaned intermediate datasets:

`experiments/data/interim/`

Audit and cleaning evidence:

`experiments/results/audits/`

`experiments/results/cleaning/`

## Outputs

Preprocessed data and partition metadata will be prepared for downstream
SE-BRL feature extraction and controlled experiments.

Generated preprocessing artifacts will be stored separately from the cleaned
source datasets so that each stage of the pipeline remains reproducible.

## Workflow

This notebook will:

1. load and verify the cleaned datasets;
2. preserve source and modality provenance;
3. establish source-supported target-label representations;
4. construct deterministic grouping identifiers for leakage prevention;
5. define modality-specific modeling datasets;
6. create grouped and stratified development, calibration, and test
   partitions;
7. prepare the development partition for grouped cross-validation;
8. define leakage-safe handling of structured missing values and other
   model-dependent transformations;
9. validate partition integrity and class distributions; and
10. export preprocessing and partition metadata for downstream stages.

SE-BRL behavioral feature extraction, classifier training, threshold
calibration, and final test evaluation are outside the scope of this notebook.

## 1. Environment and Paths

The preprocessing environment uses repository-relative paths so that the
workflow remains portable and reproducible.

Cleaned datasets are loaded from the intermediate-data directory. Raw source
files are not accessed or modified during preprocessing.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [2]:
cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / ".git").exists()
        and (p / "experiments").exists()
    ),
    None
)

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root.")

INTERIM_DIR = REPO_ROOT / "experiments" / "data" / "interim"
AUDIT_DIR = REPO_ROOT / "experiments" / "results" / "audits"
CLEANING_DIR = REPO_ROOT / "experiments" / "results" / "cleaning"

PROCESSED_DIR = REPO_ROOT / "experiments" / "data" / "processed"
PREPROCESSING_RESULTS_DIR = (
    REPO_ROOT / "experiments" / "results" / "preprocessing"
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Cleaned data:", INTERIM_DIR)
print("Audit evidence:", AUDIT_DIR)
print("Cleaning evidence:", CLEANING_DIR)
print("Processed output:", PROCESSED_DIR)
print("Preprocessing results:", PREPROCESSING_RESULTS_DIR)

Repository root: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL
Cleaned data: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/data/interim
Audit evidence: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/audits
Cleaning evidence: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/cleaning
Processed output: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/data/processed
Preprocessing results: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing


In [3]:
print("Cleaned data exists:", INTERIM_DIR.exists())
print("Audit evidence exists:", AUDIT_DIR.exists())
print("Cleaning evidence exists:", CLEANING_DIR.exists())
print("Processed output exists:", PROCESSED_DIR.exists())
print(
    "Preprocessing results exist:",
    PREPROCESSING_RESULTS_DIR.exists()
)

Cleaned data exists: True
Audit evidence exists: True
Cleaning evidence exists: True
Processed output exists: True
Preprocessing results exist: True


## 2. Load Cleaned Intermediate Datasets

The cleaned datasets produced by `02_data_cleaning.ipynb` are loaded as the
starting point for preprocessing.

At this stage, no additional transformations are applied. Dataset dimensions,
columns, and native labels are first verified to ensure that preprocessing
begins from the expected cleaned outputs.

In [4]:
DATASETS = {
    "CEAS-08":
        INTERIM_DIR / "email" / "ceas_08_clean.csv",

    "Enron":
        INTERIM_DIR / "email" / "enron_clean.csv",

    "Nazario":
        INTERIM_DIR / "email" / "nazario_clean.csv",

    "SpamAssassin":
        INTERIM_DIR / "email" / "spamassassin_clean.csv",

    "PhiUSIIL":
        INTERIM_DIR / "webpage" / "phiusiil_clean.csv",

    "UCI Phishing Websites":
        INTERIM_DIR / "webpage" / "uci_phishing_websites_clean.csv",

    "ISCX-URL2016 All":
        INTERIM_DIR / "url" / "iscx_url2016_all_clean.csv",

    "ISCX-URL2016 Phishing":
        INTERIM_DIR / "url" / "iscx_url2016_phishing_clean.csv",
}

for name, path in DATASETS.items():
    print(f"{name:26s} {'OK' if path.exists() else 'MISSING'}")

CEAS-08                    OK
Enron                      OK
Nazario                    OK
SpamAssassin               OK
PhiUSIIL                   OK
UCI Phishing Websites      OK
ISCX-URL2016 All           OK
ISCX-URL2016 Phishing      OK


In [5]:
data = {
    name: pd.read_csv(path)
    for name, path in DATASETS.items()
}

dataset_shapes = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns)
    }
    for name, df in data.items()
])

dataset_shapes

,dataset,rows,columns
0,CEAS-08,39154,7
1,Enron,29767,3
2,Nazario,1565,7
3,SpamAssassin,5809,7
4,PhiUSIIL,235795,56
5,UCI Phishing Websites,5849,31
6,ISCX-URL2016 All,26953,80
7,ISCX-URL2016 Phishing,14823,80


## 3. Native Label Semantics and Target Eligibility

The source datasets do not necessarily use equivalent definitions for their
native classes. A positive class in one dataset must therefore not be assumed
to represent the same phenomenon as a positive class in another dataset.

Before constructing experimental targets, this section records the native
label representation and determines which mappings can be supported by the
source semantics.

This distinction is particularly important for email datasets because spam,
unsolicited email, and phishing are related but are not interchangeable
classes.

No labels are changed in this section.

In [6]:
label_columns = {
    "CEAS-08": "label",
    "Enron": "label",
    "Nazario": "label",
    "SpamAssassin": "label",
    "PhiUSIIL": "label",
    "UCI Phishing Websites": "Result",
    "ISCX-URL2016 All": "URL_Type_obf_Type",
    "ISCX-URL2016 Phishing": "URL_Type_obf_Type",
}

native_label_summary = []

for name, label_col in label_columns.items():
    counts = data[name][label_col].value_counts(
        dropna=False
    )

    for value, count in counts.items():
        native_label_summary.append({
            "dataset": name,
            "label_column": label_col,
            "native_label": value,
            "count": int(count)
        })

native_label_summary = pd.DataFrame(native_label_summary)

native_label_summary

,dataset,label_column,native_label,count
0,CEAS-08,label,1,21842
1,CEAS-08,label,0,17312
2,Enron,label,0,15791
3,Enron,label,1,13976
4,Nazario,label,1,1565
5,SpamAssassin,label,0,4091
6,SpamAssassin,label,1,1718
7,PhiUSIIL,label,1,134850
8,PhiUSIIL,label,0,100945
9,UCI Phishing Websites,Result,-1,3019


### 3.1 Label Interpretation Rules

The following rules govern downstream target construction:

**PhiUSIIL**

The native binary label is preserved as supplied by the source dataset.
Its documented semantics identify `1` as legitimate and `0` as phishing.
A binary phishing target can therefore be derived by reversing the native
representation when required by an experiment.

**UCI Phishing Websites**

The native `Result` field uses `1` for legitimate observations and `-1` for
phishing observations. These values are retained in the cleaned data and may
later be mapped explicitly to a binary phishing target.

**ISCX-URL2016**

The source provides explicit categorical URL classes. `benign` and `phishing`
can therefore support a direct binary phishing-versus-benign construction.

The additional `spam`, `Defacement`, and `malware` classes are not
automatically relabeled as phishing.

`ISCX-URL2016 Phishing` is a benign/phishing subset of the broader
`ISCX-URL2016 All` data and is not treated as an independent dataset to be
concatenated with it.

**Email datasets**

The email datasets require source-specific treatment because their native
positive classes do not all represent the same phenomenon.

The Nazario corpus is a phishing-specific collection. Its observations may
therefore be used as source-supported phishing-positive examples.

CEAS-08 originates from the CEAS 2008 Live Spam Challenge. Its positive class
represents spam or unsolicited email and is not assumed to consist exclusively
of phishing messages.

The Enron dataset used in the curated release derives from an Enron spam/ham
collection. Its positive class therefore represents spam rather than a
phishing-specific ground truth.

The SpamAssassin corpus explicitly distinguishes spam from ham. Although
phishing messages may occur within its spam collection, membership in the spam
class alone does not establish that an individual message is phishing.

Accordingly, SE-BRL does not automatically map the positive classes of
CEAS-08, Enron, or SpamAssassin to phishing. Only labels whose provenance
supports the intended experimental class semantics are used as direct
phishing ground truth. No researcher-assigned or manually inferred phishing
labels are introduced.

### 3.2 Email Target Eligibility

Based on source provenance, the email datasets are separated according to the
semantic strength of their native labels.

| Dataset | Native class semantics | Direct phishing-positive eligibility |
|---|---|---|
| Nazario | Phishing corpus | Eligible |
| CEAS-08 | Spam / legitimate | Not directly eligible |
| Enron | Spam / ham | Not directly eligible |
| SpamAssassin | Spam / ham | Not directly eligible |

This distinction prevents the experimental target from treating the broader
category of spam as equivalent to phishing.

Datasets that are not eligible as direct phishing-positive ground truth may
still provide legitimate or ham observations, or may be used in explicitly
defined auxiliary experiments where their original class semantics are
preserved.

Final experimental inclusion is determined before partition construction so
that target semantics remain fixed prior to model development.

In [7]:
email_label_semantics = pd.DataFrame([
    {
        "dataset": "Nazario",
        "native_positive_semantics": "phishing",
        "phishing_positive_eligible": True
    },
    {
        "dataset": "CEAS-08",
        "native_positive_semantics": "spam",
        "phishing_positive_eligible": False
    },
    {
        "dataset": "Enron",
        "native_positive_semantics": "spam",
        "phishing_positive_eligible": False
    },
    {
        "dataset": "SpamAssassin",
        "native_positive_semantics": "spam",
        "phishing_positive_eligible": False
    }
])

email_label_semantics

,dataset,native_positive_semantics,phishing_positive_eligible
0,Nazario,phishing,True
1,CEAS-08,spam,False
2,Enron,spam,False
3,SpamAssassin,spam,False


## 4. Experimental Dataset Construction

This section constructs modality-specific binary datasets using only
source-supported class semantics.

Target construction occurs before partition assignment so that the meaning of
the prediction target is fixed independently of model development.

For binary phishing detection, the standardized target is:

- `1` = phishing
- `0` = legitimate / benign

Rows whose native labels do not provide sufficient evidence for either class
are not automatically relabeled. They are excluded from the corresponding
binary experiment or retained only for separately defined auxiliary analyses.

No behavioral indicators or researcher-assigned labels are introduced during
target construction.

### 4.1 Email Binary Dataset

The primary email binary dataset uses the Nazario phishing corpus as the
source-supported positive class.

For the negative class, only observations explicitly belonging to the
legitimate or ham class of CEAS-08, Enron, and SpamAssassin are included.

The positive spam classes of CEAS-08, Enron, and SpamAssassin are excluded
from the primary phishing-versus-legitimate target because spam membership
alone does not establish phishing.

Source provenance is retained for every observation so that source-specific
experiments and leakage-aware grouping remain possible.

In [8]:
email_parts = []

# Phishing-positive observations
nazario = data["Nazario"].copy()
nazario["source_dataset"] = "Nazario"
nazario["target"] = 1
email_parts.append(nazario)

# Legitimate / ham negative observations only
for source in ["CEAS-08", "Enron", "SpamAssassin"]:
    df = data[source].copy()

    legitimate = df[df["label"] == 0].copy()

    legitimate["source_dataset"] = source
    legitimate["target"] = 0

    email_parts.append(legitimate)

email_binary = pd.concat(
    email_parts,
    ignore_index=True,
    sort=False
)

print("Email binary rows:", len(email_binary))
print()

print("Target distribution:")
print(email_binary["target"].value_counts().sort_index())

print("\nSource × target:")
print(
    pd.crosstab(
        email_binary["source_dataset"],
        email_binary["target"]
    )
)

Email binary rows: 38759

Target distribution:
target
0    37194
1     1565
Name: count, dtype: int64

Source × target:
target              0     1
source_dataset             
CEAS-08         17312     0
Enron           15791     0
Nazario             0  1565
SpamAssassin     4091     0


#### Email Target Construction Check

The constructed email binary dataset contains 38,759 observations:

- 37,194 legitimate/ham observations (`target = 0`)
- 1,565 phishing observations (`target = 1`)

Phishing therefore represents approximately 4.04% of the constructed dataset.

This imbalance is preserved at this stage. No random undersampling,
oversampling, or synthetic sample generation is performed before partition
construction.

The source-by-target distribution also shows that phishing-positive examples
originate from Nazario, while legitimate/ham examples originate from CEAS-08,
Enron, and SpamAssassin. This source-target relationship is retained explicitly
through `source_dataset` and must be considered when designing and interpreting
the email experiments.

### 4.2 Email Experiment Framework

The email modality is evaluated through three controlled experiments
(Email-E1, Email-E2, and Email-E3).

All three experiments use the same eligible observations, partition
assignments, preprocessing controls, classifiers, and evaluation procedure.
They differ only in the feature representation specified for each experiment.

This controlled design allows differences in predictive performance to be
attributed to the evaluated representation rather than to changes in the
underlying observations or data partitions.

The common email modeling population consists of:

- source-supported phishing observations from Nazario; and
- legitimate/ham observations from CEAS-08, Enron, and SpamAssassin.

Spam-positive observations from CEAS-08, Enron, and SpamAssassin are not
silently reclassified as phishing.

A source-target dependency exists in this constructed population because the
phishing class originates from Nazario while the legitimate class originates
from the other email corpora. This relationship is preserved as provenance
metadata and must be considered when interpreting experimental results.

The exact feature definitions for Email-E1, Email-E2, and Email-E3 are applied
during feature extraction and experimentation rather than during target
construction.

### 4.3 Stable Email Observation Identifiers

Stable observation identifiers are created before partition assignment.

The identifiers are deterministic and are derived from source provenance and
the original source-row position. They are used for tracking observations
through preprocessing and experimentation and are not used as predictive
features.

In [9]:
email_binary = email_binary.copy()

email_binary["source_row_id"] = (
    email_binary
    .groupby("source_dataset")
    .cumcount()
)

email_binary["observation_id"] = (
    email_binary["source_dataset"].astype(str)
    + "::"
    + email_binary["source_row_id"].astype(str)
)

print("Rows:", len(email_binary))
print(
    "Unique observation IDs:",
    email_binary["observation_id"].nunique()
)
print(
    "Duplicate observation IDs:",
    email_binary["observation_id"].duplicated().sum()
)

email_binary[
    [
        "observation_id",
        "source_dataset",
        "source_row_id",
        "target"
    ]
].head(10)

Rows: 38759
Unique observation IDs: 38759
Duplicate observation IDs: 0


,observation_id,source_dataset,source_row_id,target
0,Nazario::0,Nazario,0,1
1,Nazario::1,Nazario,1,1
2,Nazario::2,Nazario,2,1
3,Nazario::3,Nazario,3,1
4,Nazario::4,Nazario,4,1
5,Nazario::5,Nazario,5,1
6,Nazario::6,Nazario,6,1
7,Nazario::7,Nazario,7,1
8,Nazario::8,Nazario,8,1
9,Nazario::9,Nazario,9,1


In [10]:
print("Rows:", len(email_binary))
print("Unique observation IDs:", email_binary["observation_id"].nunique())
print("Duplicate observation IDs:", email_binary["observation_id"].duplicated().sum())

Rows: 38759
Unique observation IDs: 38759
Duplicate observation IDs: 0


### 4.4 Leakage-Aware Email Grouping

Email observations that may represent the same sender identity or duplicated
message content must not be distributed across different experimental
partitions.

Grouping therefore uses two conservative relationships:

1. observations sharing the same normalized sender, where sender information is
   available; and
2. observations sharing the same normalized email body hash.

Repeated subjects alone are not treated as sufficient evidence of observation
identity because the audit found substantial subject repetition without
corresponding body duplication.

Because an observation may be connected to others through either sender or body
identity, connected components are constructed so that all linked observations
receive the same `group_id`.

The grouping variables are used only for leakage prevention and are not used as
predictive features.

In [11]:
import re

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)

    return value


def sha256_text(value):
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [12]:
email_grouping = email_binary.copy()

email_grouping["normalized_body"] = (
    email_grouping["body"]
    .fillna("")
    .map(normalize_text)
)

email_grouping["body_hash"] = (
    email_grouping["normalized_body"]
    .map(sha256_text)
)

if "sender" in email_grouping.columns:
    email_grouping["normalized_sender"] = (
        email_grouping["sender"]
        .fillna("")
        .map(normalize_text)
    )
else:
    email_grouping["normalized_sender"] = ""

In [13]:
print("Rows:", len(email_grouping))

print(
    "Rows with non-empty sender:",
    (email_grouping["normalized_sender"] != "").sum()
)

print(
    "Unique non-empty senders:",
    email_grouping.loc[
        email_grouping["normalized_sender"] != "",
        "normalized_sender"
    ].nunique()
)

print(
    "Unique body hashes:",
    email_grouping["body_hash"].nunique()
)

print(
    "Rows in duplicated body-hash groups:",
    email_grouping[
        email_grouping["body_hash"].duplicated(keep=False)
    ].shape[0]
)

Rows: 38759
Rows with non-empty sender: 22968
Unique non-empty senders: 6162
Unique body hashes: 38360
Rows in duplicated body-hash groups: 794


In [14]:
blank_body_count = (
    email_grouping["normalized_body"] == ""
).sum()

duplicated_body_groups = (
    email_grouping.loc[
        email_grouping["normalized_body"] != ""
    ]
    .groupby("body_hash")
    .size()
    .sort_values(ascending=False)
)

print("Blank normalized bodies:", blank_body_count)

print(
    "Rows in duplicated NON-EMPTY body groups:",
    duplicated_body_groups[
        duplicated_body_groups > 1
    ].sum()
)

print(
    "Number of duplicated NON-EMPTY body groups:",
    (duplicated_body_groups > 1).sum()
)

print("\nLargest duplicated body groups:")
print(
    duplicated_body_groups[
        duplicated_body_groups > 1
    ].head(10)
)

Blank normalized bodies: 2
Rows in duplicated NON-EMPTY body groups: 792
Number of duplicated NON-EMPTY body groups: 394

Largest duplicated body groups:
body_hash
200c1df91d9061a2b2b99d1e1fc5b46a9f9953ffdd753572f287dd39d78a311c    5
0486af4a5f8f827b17c60469c494ea688414af7cc518fff8923adec30da00d36    3
64d7fc9f8fdd39dbc7418b287b5b3eb9133c2f9650e4d9a01140910029f125e1    2
4ef7c0b6bf28cacaf03dbc5eaa8dd2cc55e8a18c589477128111d9a5aea9406c    2
35b1a93a1d560893f3a97e2d5c37eeb655f8c091cffe2ce443d0d20e4b6580b2    2
6cd831691c4cd31379da51ca0aa9d6b07c74f47d5c96108340f71589128f38a8    2
ac584c770c23ea2c0b90c68c308151bfc83c31414324a14fddb6ada18fda2186    2
86947a8c4f3c621fde9bfc92d209d0a27017e7a3e61975785e108ffb8bad5cd2    2
52a201a77eddf810d9ee27425fbed560592b7cd2c30d0b02ed011dba3a694991    2
e175e4127b361221fc7619bb06522db7cc907d85053cc8e3bce03dfb6f6b1fb7    2
dtype: int64


### 4.5 Connected-Component Email Groups

Leakage-prevention groups are constructed as connected components.

Two observations are linked when they share either:

- the same non-empty normalized sender; or
- the same non-empty normalized body hash.

Connections are transitive. Therefore, if one observation shares a sender with
a second observation and that second observation shares a body with a third,
all three observations are assigned to the same group.

Empty sender and body values do not create relationships.

The resulting `group_id` is used only for partition assignment and is excluded
from predictive features.

In [15]:
# Union-Find / Disjoint Set implementation
parent = list(range(len(email_grouping)))
rank = [0] * len(email_grouping)


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a == root_b:
        return

    if rank[root_a] < rank[root_b]:
        parent[root_a] = root_b

    elif rank[root_a] > rank[root_b]:
        parent[root_b] = root_a

    else:
        parent[root_b] = root_a
        rank[root_a] += 1

In [16]:
sender_groups = (
    email_grouping.loc[
        email_grouping["normalized_sender"] != ""
    ]
    .groupby("normalized_sender")
    .groups
)

for indices in sender_groups.values():
    indices = list(indices)

    first = indices[0]

    for idx in indices[1:]:
        union(first, idx)

In [17]:
body_groups = (
    email_grouping.loc[
        email_grouping["normalized_body"] != ""
    ]
    .groupby("body_hash")
    .groups
)

for indices in body_groups.values():
    indices = list(indices)

    first = indices[0]

    for idx in indices[1:]:
        union(first, idx)

In [18]:
roots = [find(i) for i in range(len(email_grouping))]

root_to_group = {
    root: f"email_group_{group_number:06d}"
    for group_number, root in enumerate(
        sorted(set(roots)),
        start=1
    )
}

email_grouping["group_id"] = [
    root_to_group[root]
    for root in roots
]

In [19]:
group_sizes = (
    email_grouping
    .groupby("group_id")
    .size()
    .sort_values(ascending=False)
)

print("Rows:", len(email_grouping))
print("Unique groups:", email_grouping["group_id"].nunique())

print(
    "Rows belonging to multi-row groups:",
    group_sizes[group_sizes > 1].sum()
)

print(
    "Number of multi-row groups:",
    (group_sizes > 1).sum()
)

print(
    "Largest group size:",
    group_sizes.max()
)

print("\nLargest groups:")
print(group_sizes.head(10))

Rows: 38759
Unique groups: 21601
Rows belonging to multi-row groups: 19887
Number of multi-row groups: 2729
Largest group size: 462

Largest groups:
group_id
email_group_001411    462
email_group_001825    295
email_group_002116    276
email_group_001605    208
email_group_001413    183
email_group_001455    158
email_group_002036    152
email_group_002071    131
email_group_001450    124
email_group_001612    113
dtype: int64


### 4.6 Email Group Integrity Check

Before partition construction, the connected-component groups are evaluated for
target and source composition.

A valid leakage-prevention group may contain multiple observations and multiple
sources. However, any group containing both phishing and legitimate targets
must be identified explicitly because the complete connected component must
remain within a single partition.

This check does not modify labels or dissolve groups.

In [20]:
group_integrity = (
    email_grouping
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        target_classes=("target", "nunique"),
        source_count=("source_dataset", "nunique")
    )
    .reset_index()
)

mixed_target_groups = group_integrity[
    group_integrity["target_classes"] > 1
]

multi_source_groups = group_integrity[
    group_integrity["source_count"] > 1
]

print("Total groups:", len(group_integrity))

print(
    "Mixed-target groups:",
    len(mixed_target_groups)
)

print(
    "Rows inside mixed-target groups:",
    email_grouping[
        email_grouping["group_id"].isin(
            mixed_target_groups["group_id"]
        )
    ].shape[0]
)

print(
    "Multi-source groups:",
    len(multi_source_groups)
)

print(
    "Rows inside multi-source groups:",
    email_grouping[
        email_grouping["group_id"].isin(
            multi_source_groups["group_id"]
        )
    ].shape[0]
)

Total groups: 21601
Mixed-target groups: 0
Rows inside mixed-target groups: 0
Multi-source groups: 0
Rows inside multi-source groups: 0


In [21]:
mixed_target_summary = (
    email_grouping[
        email_grouping["group_id"].isin(
            mixed_target_groups["group_id"]
        )
    ]
    .groupby(
        [
            "group_id",
            "source_dataset",
            "target"
        ]
    )
    .size()
    .reset_index(name="rows")
)

mixed_target_summary

,group_id,source_dataset,target,rows


#### Email Group Integrity Result

No connected-component group contained both phishing and legitimate targets.

This indicates that the leakage-prevention grouping strategy did not create
target-conflicting components. The resulting groups may therefore be used for
partition construction while preserving the requirement that related
observations remain within the same experimental partition.

Group identifiers remain metadata only and are not used as predictive
features.

In [22]:
multi_source_summary = (
    email_grouping[
        email_grouping["group_id"].isin(
            multi_source_groups["group_id"]
        )
    ]
    .groupby(
        [
            "group_id",
            "source_dataset",
            "target"
        ]
    )
    .size()
    .reset_index(name="rows")
)

print(
    "Multi-source groups:",
    multi_source_summary["group_id"].nunique()
)

multi_source_summary

Multi-source groups: 0


,group_id,source_dataset,target,rows


#### Multi-Source Group Check

No connected-component group contained observations from more than one source
dataset within the constructed email modeling population.

This indicates that the sender and body-based grouping relationships present
after target eligibility filtering are source-local.

This result does not alter the grouping strategy. Exact non-empty body identity
and sender identity remain the leakage-prevention criteria, and all resulting
groups are preserved intact during partition assignment.

In [23]:
email_binary = email_grouping.copy()

print("Rows:", len(email_binary))
print("Groups:", email_binary["group_id"].nunique())
print("Targets:", email_binary["target"].value_counts().to_dict())

Rows: 38759
Groups: 21601
Targets: {0: 37194, 1: 1565}


## 5. Grouped and Stratified Email Partitioning

The email modeling population is partitioned into development, calibration,
and sealed test subsets using the predefined 70/15/15 design.

Partition construction follows two requirements:

1. every leakage-prevention `group_id` must remain entirely within one
   partition; and
2. target and source composition should remain approximately proportional
   across partitions.

Because group sizes vary, exact row percentages cannot be guaranteed without
breaking groups. Partition assignment therefore prioritizes group integrity
while minimizing deviation from the intended 70/15/15 row distribution.

The development partition is used for model development and grouped
cross-validation. The calibration partition is reserved for threshold
selection. The test partition remains sealed until final evaluation.

In [24]:
PARTITION_RATIOS = {
    "development": 0.70,
    "calibration": 0.15,
    "test": 0.15
}

RANDOM_SEED = 42

In [25]:
group_table = (
    email_binary
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        target=("target", "first"),
        source_dataset=("source_dataset", "first")
    )
    .reset_index()
)

print("Groups:", len(group_table))
print("Rows represented:", group_table["rows"].sum())

group_table.head()

Groups: 21601
Rows represented: 38759


,group_id,rows,target,source_dataset
0,email_group_000001,2,1,Nazario
1,email_group_000002,1,1,Nazario
2,email_group_000003,1,1,Nazario
3,email_group_000004,1,1,Nazario
4,email_group_000005,1,1,Nazario


In [26]:
rng = np.random.default_rng(RANDOM_SEED)

partition_assignments = []

for (source, target), stratum in group_table.groupby(
    ["source_dataset", "target"]
):
    stratum = stratum.copy()

    # Random tie-breaking before sorting large groups first
    stratum["_random"] = rng.random(len(stratum))

    stratum = stratum.sort_values(
        ["rows", "_random"],
        ascending=[False, True]
    )

    total_rows = stratum["rows"].sum()

    target_rows = {
        partition: total_rows * ratio
        for partition, ratio in PARTITION_RATIOS.items()
    }

    assigned_rows = {
        partition: 0
        for partition in PARTITION_RATIOS
    }

    for _, row in stratum.iterrows():

        deficits = {
            partition: (
                target_rows[partition]
                - assigned_rows[partition]
            ) / max(target_rows[partition], 1)
            for partition in PARTITION_RATIOS
        }

        chosen_partition = max(
            deficits,
            key=deficits.get
        )

        partition_assignments.append({
            "group_id": row["group_id"],
            "partition": chosen_partition
        })

        assigned_rows[chosen_partition] += row["rows"]

In [27]:
partition_map = pd.DataFrame(partition_assignments)

email_binary = email_binary.merge(
    partition_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    email_binary["partition"]
    .value_counts()
)

print()

print(
    email_binary["partition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

partition
development    27129
test            5815
calibration     5815
Name: count, dtype: int64

partition
development    69.99
test           15.00
calibration    15.00
Name: proportion, dtype: float64


### 5.1 Partition Integrity Validation

The generated partitions are validated before any model-dependent
preprocessing is performed.

Validation checks confirm that:

- every observation is assigned to exactly one partition;
- no `group_id` appears in more than one partition;
- phishing observations are present in all partitions;
- class prevalence remains approximately consistent across partitions; and
- source composition remains approximately proportional across partitions.

These checks ensure that the partitioning procedure preserves both leakage
controls and the intended experimental design.

In [28]:
# 1. Every row must have exactly one partition
print("Total rows:", len(email_binary))
print("Rows with missing partition:", email_binary["partition"].isna().sum())

# 2. No group may cross partitions
group_partition_counts = (
    email_binary
    .groupby("group_id")["partition"]
    .nunique()
)

print(
    "Groups appearing in multiple partitions:",
    (group_partition_counts > 1).sum()
)

# 3. Partition × target distribution
partition_target_counts = pd.crosstab(
    email_binary["partition"],
    email_binary["target"]
)

print("\nPartition × target counts:")
print(partition_target_counts)

# 4. Target prevalence
partition_target_prevalence = (
    email_binary
    .groupby("partition")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(partition_target_prevalence)

# 5. Source composition
partition_source_counts = pd.crosstab(
    email_binary["partition"],
    email_binary["source_dataset"]
)

print("\nPartition × source counts:")
print(partition_source_counts)

Total rows: 38759
Rows with missing partition: 0
Groups appearing in multiple partitions: 0

Partition × target counts:
target           0     1
partition               
calibration   5580   235
development  26034  1095
test          5580   235

Phishing prevalence (%):
partition
calibration    4.04
development    4.04
test           4.04
Name: target, dtype: float64

Partition × source counts:
source_dataset  CEAS-08  Enron  Nazario  SpamAssassin
partition                                            
calibration        2597   2369      235           614
development       12118  11053     1095          2863
test               2597   2369      235           614


In [29]:
assert len(email_binary) == 38759
assert email_binary["partition"].isna().sum() == 0
assert (group_partition_counts > 1).sum() == 0

for partition in [
    "development",
    "calibration",
    "test"
]:
    assert (
        email_binary.loc[
            email_binary["partition"] == partition,
            "target"
        ].nunique()
        == 2
    )

print("All email partition integrity checks passed.")

All email partition integrity checks passed.


#### Email Partition Validation Result

The email dataset was successfully partitioned into development, calibration,
and test subsets while preserving leakage-prevention groups.

No observation was left unassigned, and no `group_id` appeared in more than
one partition.

Class prevalence remained stable at approximately 4.04% phishing in all three
partitions. Source composition was also preserved proportionally across the
development, calibration, and test subsets.

These results confirm that the 70/15/15 partitioning procedure maintains the
intended experimental distribution without breaking leakage-prevention groups.

The test partition is treated as sealed from this point onward and is not used
for model development, feature-selection decisions, or threshold calibration.

## 6. Grouped Cross-Validation Within the Development Partition

Model development uses five-fold grouped cross-validation applied only to the
development partition.

The calibration and test partitions are excluded from cross-validation.

Cross-validation preserves the same leakage-prevention `group_id` constraints
used during the outer partitioning procedure. Therefore, observations belonging
to the same connected email group cannot appear in both the training and
validation portions of a fold.

Fold assignment is also constructed to preserve target and source composition
as closely as possible.

Any learned preprocessing transformation used during model development must be
fitted separately within each training fold and then applied to the
corresponding validation fold.

In [30]:
email_dev = (
    email_binary[
        email_binary["partition"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Development rows:", len(email_dev))
print("Development groups:", email_dev["group_id"].nunique())
print(
    "Development target distribution:",
    email_dev["target"].value_counts().to_dict()
)

Development rows: 27129
Development groups: 15125
Development target distribution: {0: 26034, 1: 1095}


In [31]:
dev_group_table = (
    email_dev
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        target=("target", "first"),
        source_dataset=("source_dataset", "first")
    )
    .reset_index()
)

print("Development groups:", len(dev_group_table))
print("Rows represented:", dev_group_table["rows"].sum())

Development groups: 15125
Rows represented: 27129


In [32]:
N_FOLDS = 5
CV_RANDOM_SEED = 42

cv_rng = np.random.default_rng(CV_RANDOM_SEED)

fold_assignments = []

for (source, target), stratum in dev_group_table.groupby(
    ["source_dataset", "target"]
):
    stratum = stratum.copy()

    stratum["_random"] = cv_rng.random(len(stratum))

    stratum = stratum.sort_values(
        ["rows", "_random"],
        ascending=[False, True]
    )

    total_rows = stratum["rows"].sum()
    target_rows_per_fold = total_rows / N_FOLDS

    assigned_rows = {
        fold: 0
        for fold in range(1, N_FOLDS + 1)
    }

    for _, row in stratum.iterrows():

        chosen_fold = min(
            assigned_rows,
            key=assigned_rows.get
        )

        fold_assignments.append({
            "group_id": row["group_id"],
            "cv_fold": chosen_fold
        })

        assigned_rows[chosen_fold] += row["rows"]

In [33]:
cv_fold_map = pd.DataFrame(fold_assignments)

email_dev = email_dev.merge(
    cv_fold_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    email_dev["cv_fold"]
    .value_counts()
    .sort_index()
)

print()

print(
    email_dev["cv_fold"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

cv_fold
1    5427
2    5427
3    5427
4    5424
5    5424
Name: count, dtype: int64

cv_fold
1    20.00
2    20.00
3    20.00
4    19.99
5    19.99
Name: proportion, dtype: float64


### 6.1 Cross-Validation Fold Integrity Validation

The five development folds are validated to ensure that leakage-prevention
groups remain intact and that each fold preserves the target and source
distribution as closely as possible.

Validation confirms that:

- every development observation receives exactly one fold assignment;
- no `group_id` appears in more than one fold;
- every fold contains both phishing and legitimate observations;
- phishing prevalence remains approximately stable across folds; and
- source composition remains approximately proportional across folds.

In [34]:
# 1. Every development row must have one fold
print("Development rows:", len(email_dev))
print("Rows with missing fold:", email_dev["cv_fold"].isna().sum())

# 2. No group may cross folds
group_fold_counts = (
    email_dev
    .groupby("group_id")["cv_fold"]
    .nunique()
)

print(
    "Groups appearing in multiple folds:",
    (group_fold_counts > 1).sum()
)

# 3. Fold × target counts
fold_target_counts = pd.crosstab(
    email_dev["cv_fold"],
    email_dev["target"]
)

print("\nFold × target counts:")
print(fold_target_counts)

# 4. Phishing prevalence per fold
fold_target_prevalence = (
    email_dev
    .groupby("cv_fold")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(fold_target_prevalence)

# 5. Source composition per fold
fold_source_counts = pd.crosstab(
    email_dev["cv_fold"],
    email_dev["source_dataset"]
)

print("\nFold × source counts:")
print(fold_source_counts)

Development rows: 27129
Rows with missing fold: 0
Groups appearing in multiple folds: 0

Fold × target counts:
target      0    1
cv_fold           
1        5208  219
2        5208  219
3        5208  219
4        5205  219
5        5205  219

Phishing prevalence (%):
cv_fold
1    4.04
2    4.04
3    4.04
4    4.04
5    4.04
Name: target, dtype: float64

Fold × source counts:
source_dataset  CEAS-08  Enron  Nazario  SpamAssassin
cv_fold                                              
1                  2424   2211      219           573
2                  2424   2211      219           573
3                  2424   2211      219           573
4                  2423   2210      219           572
5                  2423   2210      219           572


In [35]:
assert len(email_dev) == 27129
assert email_dev["cv_fold"].isna().sum() == 0
assert (group_fold_counts > 1).sum() == 0

for fold in range(1, 6):
    assert (
        email_dev.loc[
            email_dev["cv_fold"] == fold,
            "target"
        ].nunique()
        == 2
    )

print("All grouped cross-validation integrity checks passed.")

All grouped cross-validation integrity checks passed.


#### Grouped Cross-Validation Result

The development partition was successfully divided into five approximately
equal folds while preserving leakage-prevention groups.

No `group_id` crossed fold boundaries, and each fold retained both target
classes. Class prevalence and source composition remained approximately
consistent across folds.

These folds may therefore be used for model development, with any learned
preprocessing step fitted independently within each training fold and applied
only to its corresponding validation fold.

## 7. PhiUSIIL Webpage Dataset Construction

PhiUSIIL provides an explicit binary label and raw URL/domain information.

The documented native label semantics are:

- `0` = phishing
- `1` = legitimate

For SE-BRL, the standardized binary target is defined as:

- `1` = phishing
- `0` = legitimate

The native label is preserved separately, and the standardized target is
derived deterministically from the documented source semantics.

Domain identity is retained for leakage-aware grouping so that observations
from the same domain do not appear in different experimental partitions.

In [36]:
phiusiil = data["PhiUSIIL"].copy()

phiusiil["source_dataset"] = "PhiUSIIL"
phiusiil["source_row_id"] = np.arange(len(phiusiil))

phiusiil["observation_id"] = (
    phiusiil["source_dataset"]
    + "::"
    + phiusiil["source_row_id"].astype(str)
)

# Standardized target:
# native 0 = phishing  -> target 1
# native 1 = legitimate -> target 0
phiusiil["target"] = (
    phiusiil["label"]
    .map({
        0: 1,
        1: 0
    })
)

In [37]:
print("Rows:", len(phiusiil))
print("Unique observation IDs:", phiusiil["observation_id"].nunique())
print("Missing targets:", phiusiil["target"].isna().sum())

print("\nNative label distribution:")
print(
    phiusiil["label"]
    .value_counts()
    .sort_index()
)

print("\nStandardized target distribution:")
print(
    phiusiil["target"]
    .value_counts()
    .sort_index()
)

Rows: 235795
Unique observation IDs: 235795
Missing targets: 0

Native label distribution:
label
0    100945
1    134850
Name: count, dtype: int64

Standardized target distribution:
target
0    134850
1    100945
Name: count, dtype: int64


### 7.1 Domain-Based Leakage Grouping

PhiUSIIL observations are grouped by normalized domain.

This prevents observations associated with the same domain from appearing in
different development, calibration, or test partitions.

The grouping variable is used only for leakage prevention and is excluded from
predictive features.

In [38]:
phiusiil["normalized_domain"] = (
    phiusiil["Domain"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Blank domains:",
    (phiusiil["normalized_domain"] == "").sum()
)

print(
    "Unique non-empty domains:",
    phiusiil.loc[
        phiusiil["normalized_domain"] != "",
        "normalized_domain"
    ].nunique()
)

Blank domains: 0
Unique non-empty domains: 220086


In [39]:
domain_values = sorted(
    phiusiil["normalized_domain"].unique()
)

domain_to_group = {
    domain: f"phiusiil_domain_{i:06d}"
    for i, domain in enumerate(
        domain_values,
        start=1
    )
}

phiusiil["group_id"] = (
    phiusiil["normalized_domain"]
    .map(domain_to_group)
)

In [40]:
phiusiil_group_summary = (
    phiusiil
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        target_classes=("target", "nunique")
    )
    .reset_index()
)

print(
    "Total groups:",
    len(phiusiil_group_summary)
)

print(
    "Multi-row groups:",
    (phiusiil_group_summary["rows"] > 1).sum()
)

print(
    "Largest group size:",
    phiusiil_group_summary["rows"].max()
)

print(
    "Mixed-target groups:",
    (phiusiil_group_summary["target_classes"] > 1).sum()
)

Total groups: 220086
Multi-row groups: 5526
Largest group size: 1197
Mixed-target groups: 54


### 7.2 Mixed-Target Domain Handling

A small number of normalized domains contain observations from both target
classes.

These mixed-target groups are not removed, split, or manually corrected.

Instead, the complete domain group is kept intact during partition assignment.
Partition balancing is therefore based on the number of phishing and legitimate
observations contributed by each domain group.

This preserves domain-level leakage prevention without introducing
researcher-assigned label decisions.

In [41]:
phiusiil_group_table = (
    phiusiil
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        legitimate_rows=("target", lambda s: int((s == 0).sum())),
        phishing_rows=("target", lambda s: int((s == 1).sum()))
    )
    .reset_index()
)

print("Groups:", len(phiusiil_group_table))
print("Rows represented:", phiusiil_group_table["rows"].sum())

print(
    "Groups containing both targets:",
    (
        (phiusiil_group_table["legitimate_rows"] > 0)
        &
        (phiusiil_group_table["phishing_rows"] > 0)
    ).sum()
)

Groups: 220086
Rows represented: 235795
Groups containing both targets: 54


### 7.3 Grouped PhiUSIIL Partitioning

PhiUSIIL is partitioned into development, calibration, and test subsets using
the predefined 70/15/15 design.

All observations sharing the same normalized domain remain within the same
partition.

Because some domain groups contain both target classes, partition assignment
is based on group-level counts of:

- total observations;
- legitimate observations; and
- phishing observations.

The assignment procedure seeks to minimize deviation from the intended
70/15/15 distribution across all three quantities while preserving complete
domain groups.

The calibration and test partitions are not used for model development.

In [42]:
PHIUSIIL_PARTITION_RATIOS = {
    "development": 0.70,
    "calibration": 0.15,
    "test": 0.15
}

PHIUSIIL_RANDOM_SEED = 42

In [43]:
rng = np.random.default_rng(PHIUSIIL_RANDOM_SEED)

phiusiil_groups = phiusiil_group_table.copy()

phiusiil_groups["_random"] = rng.random(
    len(phiusiil_groups)
)

# Large and compositionally important groups first
phiusiil_groups = phiusiil_groups.sort_values(
    ["rows", "phishing_rows", "legitimate_rows", "_random"],
    ascending=[False, False, False, True]
)

totals = {
    "rows": phiusiil_groups["rows"].sum(),
    "legitimate_rows": phiusiil_groups["legitimate_rows"].sum(),
    "phishing_rows": phiusiil_groups["phishing_rows"].sum()
}

targets = {
    partition: {
        metric: totals[metric] * ratio
        for metric in totals
    }
    for partition, ratio in PHIUSIIL_PARTITION_RATIOS.items()
}

assigned = {
    partition: {
        "rows": 0,
        "legitimate_rows": 0,
        "phishing_rows": 0
    }
    for partition in PHIUSIIL_PARTITION_RATIOS
}

phiusiil_partition_assignments = []

for _, row in phiusiil_groups.iterrows():

    best_partition = None
    best_score = None

    for partition in PHIUSIIL_PARTITION_RATIOS:

        score = 0.0

        for metric in [
            "rows",
            "legitimate_rows",
            "phishing_rows"
        ]:
            projected = (
                assigned[partition][metric]
                + row[metric]
            )

            target_value = targets[partition][metric]

            score += (
                (projected - target_value)
                / max(target_value, 1)
            ) ** 2

        if best_score is None or score < best_score:
            best_score = score
            best_partition = partition

    phiusiil_partition_assignments.append({
        "group_id": row["group_id"],
        "partition": best_partition
    })

    for metric in [
        "rows",
        "legitimate_rows",
        "phishing_rows"
    ]:
        assigned[best_partition][metric] += row[metric]

In [44]:
phiusiil_partition_map = pd.DataFrame(
    phiusiil_partition_assignments
)

phiusiil = phiusiil.merge(
    phiusiil_partition_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    phiusiil["partition"]
    .value_counts()
)

print()

print(
    phiusiil["partition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

partition
development    162616
calibration     36630
test            36549
Name: count, dtype: int64

partition
development    68.96
calibration    15.53
test           15.50
Name: proportion, dtype: float64


In [45]:
rng = np.random.default_rng(PHIUSIIL_RANDOM_SEED)

phiusiil_groups = phiusiil_group_table.copy()

phiusiil_groups["_random"] = rng.random(
    len(phiusiil_groups)
)

# Assign larger groups first
phiusiil_groups = phiusiil_groups.sort_values(
    ["rows", "_random"],
    ascending=[False, True]
)

metrics = [
    "rows",
    "legitimate_rows",
    "phishing_rows"
]

totals = {
    metric: phiusiil_groups[metric].sum()
    for metric in metrics
}

targets = {
    partition: {
        metric: totals[metric] * ratio
        for metric in metrics
    }
    for partition, ratio in PHIUSIIL_PARTITION_RATIOS.items()
}

assigned = {
    partition: {
        metric: 0
        for metric in metrics
    }
    for partition in PHIUSIIL_PARTITION_RATIOS
}

phiusiil_partition_assignments = []

for _, row in phiusiil_groups.iterrows():

    partition_scores = {}

    for partition in PHIUSIIL_PARTITION_RATIOS:

        relative_deficits = []

        for metric in metrics:

            target_value = targets[partition][metric]

            remaining = (
                target_value
                - assigned[partition][metric]
            )

            relative_deficit = (
                remaining / max(target_value, 1)
            )

            relative_deficits.append(
                relative_deficit
            )

        # Prefer the partition that is most under-filled
        partition_scores[partition] = np.mean(
            relative_deficits
        )

    chosen_partition = max(
        partition_scores,
        key=partition_scores.get
    )

    phiusiil_partition_assignments.append({
        "group_id": row["group_id"],
        "partition": chosen_partition
    })

    for metric in metrics:
        assigned[chosen_partition][metric] += row[metric]

In [46]:
phiusiil = data["PhiUSIIL"].copy()

phiusiil["source_dataset"] = "PhiUSIIL"
phiusiil["source_row_id"] = np.arange(len(phiusiil))

phiusiil["observation_id"] = (
    phiusiil["source_dataset"]
    + "::"
    + phiusiil["source_row_id"].astype(str)
)

phiusiil["target"] = (
    phiusiil["label"]
    .map({
        0: 1,
        1: 0
    })
)

phiusiil["normalized_domain"] = (
    phiusiil["Domain"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

phiusiil["group_id"] = (
    phiusiil["normalized_domain"]
    .map(domain_to_group)
)

In [47]:
phiusiil_partition_map = pd.DataFrame(
    phiusiil_partition_assignments
)

phiusiil = phiusiil.merge(
    phiusiil_partition_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    phiusiil["partition"]
    .value_counts()
)

print()

print(
    phiusiil["partition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

partition
development    165089
calibration     35358
test            35348
Name: count, dtype: int64

partition
development    70.01
calibration    15.00
test           14.99
Name: proportion, dtype: float64


### 7.4 PhiUSIIL Partition Integrity Validation

The grouped PhiUSIIL partitions are validated to confirm that domain-level
leakage prevention and target distributions are preserved.

Validation confirms that:

- every observation is assigned to exactly one partition;
- no domain-based `group_id` appears in more than one partition;
- both target classes are represented in all partitions;
- phishing prevalence remains approximately consistent across partitions; and
- mixed-target domains remain intact within a single partition.

The test partition is treated as sealed after validation.

In [48]:
print("Total rows:", len(phiusiil))
print(
    "Rows with missing partition:",
    phiusiil["partition"].isna().sum()
)

group_partition_counts = (
    phiusiil
    .groupby("group_id")["partition"]
    .nunique()
)

print(
    "Groups appearing in multiple partitions:",
    (group_partition_counts > 1).sum()
)

partition_target_counts = pd.crosstab(
    phiusiil["partition"],
    phiusiil["target"]
)

print("\nPartition × target counts:")
print(partition_target_counts)

partition_target_prevalence = (
    phiusiil
    .groupby("partition")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(partition_target_prevalence)

Total rows: 235795
Rows with missing partition: 0
Groups appearing in multiple partitions: 0

Partition × target counts:
target           0      1
partition                
calibration  20165  15193
development  94582  70507
test         20103  15245

Phishing prevalence (%):
partition
calibration    42.97
development    42.71
test           43.13
Name: target, dtype: float64


In [49]:
mixed_phiusiil_groups = phiusiil_group_table.loc[
    (phiusiil_group_table["legitimate_rows"] > 0)
    &
    (phiusiil_group_table["phishing_rows"] > 0),
    "group_id"
]

mixed_group_partition_counts = (
    phiusiil[
        phiusiil["group_id"].isin(
            mixed_phiusiil_groups
        )
    ]
    .groupby("group_id")["partition"]
    .nunique()
)

print(
    "Mixed-target groups crossing partitions:",
    (mixed_group_partition_counts > 1).sum()
)

Mixed-target groups crossing partitions: 0


In [50]:
assert len(phiusiil) == 235795
assert phiusiil["partition"].isna().sum() == 0
assert (group_partition_counts > 1).sum() == 0
assert (mixed_group_partition_counts > 1).sum() == 0

for partition in [
    "development",
    "calibration",
    "test"
]:
    assert (
        phiusiil.loc[
            phiusiil["partition"] == partition,
            "target"
        ].nunique()
        == 2
    )

print("All PhiUSIIL partition integrity checks passed.")

All PhiUSIIL partition integrity checks passed.


#### PhiUSIIL Partition Validation Result

The PhiUSIIL dataset was successfully partitioned into development,
calibration, and test subsets while preserving domain-level leakage controls.

No domain-based `group_id` appeared in more than one partition, and all
mixed-target domain groups remained intact.

Phishing prevalence remained similar across partitions, ranging from
approximately 42.71% to 43.13%. The small variation reflects the requirement
to preserve complete domain groups rather than split them solely to obtain
perfect class proportions.

The resulting partitions satisfy the intended 70/15/15 experimental design.
The test partition is treated as sealed from this point onward.

## 8. Grouped Cross-Validation for PhiUSIIL

Five-fold grouped cross-validation is constructed using only the PhiUSIIL
development partition.

Domain-based `group_id` values remain intact across folds so that observations
from the same domain cannot occur in both the training and validation portions
of a fold.

Because some domain groups contain observations from both target classes,
fold assignment accounts for:

- total observations;
- legitimate observations; and
- phishing observations.

The calibration and sealed test partitions are excluded from cross-validation.

In [51]:
phiusiil_dev = (
    phiusiil[
        phiusiil["partition"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Development rows:", len(phiusiil_dev))
print("Development groups:", phiusiil_dev["group_id"].nunique())
print(
    "Target distribution:",
    phiusiil_dev["target"].value_counts().to_dict()
)

Development rows: 165089
Development groups: 154096
Target distribution: {0: 94582, 1: 70507}


In [52]:
phiusiil_dev_group_table = (
    phiusiil_dev
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        legitimate_rows=("target", lambda s: int((s == 0).sum())),
        phishing_rows=("target", lambda s: int((s == 1).sum()))
    )
    .reset_index()
)

print(
    "Development groups:",
    len(phiusiil_dev_group_table)
)

print(
    "Rows represented:",
    phiusiil_dev_group_table["rows"].sum()
)

Development groups: 154096
Rows represented: 165089


In [53]:
PHIUSIIL_N_FOLDS = 5
PHIUSIIL_CV_RANDOM_SEED = 42

cv_rng = np.random.default_rng(
    PHIUSIIL_CV_RANDOM_SEED
)

cv_groups = phiusiil_dev_group_table.copy()

cv_groups["_random"] = cv_rng.random(
    len(cv_groups)
)

cv_groups = cv_groups.sort_values(
    ["rows", "_random"],
    ascending=[False, True]
)

metrics = [
    "rows",
    "legitimate_rows",
    "phishing_rows"
]

totals = {
    metric: cv_groups[metric].sum()
    for metric in metrics
}

targets_per_fold = {
    fold: {
        metric: totals[metric] / PHIUSIIL_N_FOLDS
        for metric in metrics
    }
    for fold in range(1, PHIUSIIL_N_FOLDS + 1)
}

assigned = {
    fold: {
        metric: 0
        for metric in metrics
    }
    for fold in range(1, PHIUSIIL_N_FOLDS + 1)
}

phiusiil_fold_assignments = []

for _, row in cv_groups.iterrows():

    fold_scores = {}

    for fold in range(1, PHIUSIIL_N_FOLDS + 1):

        relative_deficits = []

        for metric in metrics:

            target_value = targets_per_fold[fold][metric]

            remaining = (
                target_value
                - assigned[fold][metric]
            )

            relative_deficit = (
                remaining
                / max(target_value, 1)
            )

            relative_deficits.append(
                relative_deficit
            )

        fold_scores[fold] = np.mean(
            relative_deficits
        )

    chosen_fold = max(
        fold_scores,
        key=fold_scores.get
    )

    phiusiil_fold_assignments.append({
        "group_id": row["group_id"],
        "cv_fold": chosen_fold
    })

    for metric in metrics:
        assigned[chosen_fold][metric] += row[metric]

In [54]:
phiusiil_fold_map = pd.DataFrame(
    phiusiil_fold_assignments
)

phiusiil_dev = phiusiil_dev.merge(
    phiusiil_fold_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    phiusiil_dev["cv_fold"]
    .value_counts()
    .sort_index()
)

print()

print(
    phiusiil_dev["cv_fold"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

cv_fold
1    32973
2    33052
3    33031
4    33016
5    33017
Name: count, dtype: int64

cv_fold
1    19.97
2    20.02
3    20.01
4    20.00
5    20.00
Name: proportion, dtype: float64


### 8.1 PhiUSIIL Cross-Validation Integrity Validation

The five PhiUSIIL development folds are validated to confirm that domain-level
leakage prevention and target distributions are preserved.

Validation confirms that:

- every development observation receives exactly one fold assignment;
- no domain-based `group_id` appears in more than one fold;
- both target classes are present in every fold;
- phishing prevalence remains approximately stable across folds; and
- mixed-target domain groups remain intact within a single fold.

In [55]:
print("Development rows:", len(phiusiil_dev))
print(
    "Rows with missing fold:",
    phiusiil_dev["cv_fold"].isna().sum()
)

group_fold_counts = (
    phiusiil_dev
    .groupby("group_id")["cv_fold"]
    .nunique()
)

print(
    "Groups appearing in multiple folds:",
    (group_fold_counts > 1).sum()
)

fold_target_counts = pd.crosstab(
    phiusiil_dev["cv_fold"],
    phiusiil_dev["target"]
)

print("\nFold × target counts:")
print(fold_target_counts)

fold_target_prevalence = (
    phiusiil_dev
    .groupby("cv_fold")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(fold_target_prevalence)

Development rows: 165089
Rows with missing fold: 0
Groups appearing in multiple folds: 0

Fold × target counts:
target       0      1
cv_fold              
1        18664  14309
2        19111  13941
3        18992  14039
4        18905  14111
5        18910  14107

Phishing prevalence (%):
cv_fold
1    43.40
2    42.18
3    42.50
4    42.74
5    42.73
Name: target, dtype: float64


In [56]:
dev_mixed_groups = (
    phiusiil_dev_group_table.loc[
        (phiusiil_dev_group_table["legitimate_rows"] > 0)
        &
        (phiusiil_dev_group_table["phishing_rows"] > 0),
        "group_id"
    ]
)

mixed_group_fold_counts = (
    phiusiil_dev[
        phiusiil_dev["group_id"].isin(
            dev_mixed_groups
        )
    ]
    .groupby("group_id")["cv_fold"]
    .nunique()
)

print(
    "Mixed-target groups crossing folds:",
    (mixed_group_fold_counts > 1).sum()
)

Mixed-target groups crossing folds: 0


In [57]:
assert len(phiusiil_dev) == 165089
assert phiusiil_dev["cv_fold"].isna().sum() == 0
assert (group_fold_counts > 1).sum() == 0
assert (mixed_group_fold_counts > 1).sum() == 0

for fold in range(1, 6):
    assert (
        phiusiil_dev.loc[
            phiusiil_dev["cv_fold"] == fold,
            "target"
        ].nunique()
        == 2
    )

print(
    "All PhiUSIIL grouped cross-validation "
    "integrity checks passed."
)

All PhiUSIIL grouped cross-validation integrity checks passed.


## 9. UCI Phishing Websites Dataset Construction

The UCI Phishing Websites dataset provides structured phishing indicators but
does not include the original webpage URL, domain, or another direct observation
identifier.

The documented native target semantics are:

- `1` = legitimate
- `-1` = phishing

For SE-BRL, the standardized target is defined as:

- `1` = phishing
- `0` = legitimate

Because no raw URL/domain identity is available, leakage prevention is based on
the complete predictor feature vector excluding the native `Result` label.

Rows with identical predictor vectors are assigned to the same group even when
their native labels differ. Such conflicting groups are preserved without
manual correction or researcher-assigned relabeling.

In [58]:
uci = data["UCI Phishing Websites"].copy()

uci["source_dataset"] = "UCI Phishing Websites"
uci["source_row_id"] = np.arange(len(uci))

uci["observation_id"] = (
    "UCI::"
    + uci["source_row_id"].astype(str)
)

uci["target"] = (
    uci["Result"]
    .map({
        -1: 1,
         1: 0
    })
)

print("Rows:", len(uci))
print("Unique observation IDs:", uci["observation_id"].nunique())
print("Missing targets:", uci["target"].isna().sum())

print("\nNative labels:")
print(uci["Result"].value_counts().sort_index())

print("\nStandardized targets:")
print(uci["target"].value_counts().sort_index())

Rows: 5849
Unique observation IDs: 5849
Missing targets: 0

Native labels:
Result
-1    3019
 1    2830
Name: count, dtype: int64

Standardized targets:
target
0    2830
1    3019
Name: count, dtype: int64


### 9.1 Feature-Vector Leakage Grouping

UCI observations are grouped by the complete predictor vector excluding the
native target column.

This provides a conservative leakage-control strategy when raw URL or domain
identity is unavailable.

Identical predictor vectors are kept within a single partition, including
groups where the same predictor vector is associated with conflicting native
labels.

The resulting group identifier is used only for partition assignment and is
not used as a predictive feature.

In [59]:
uci_predictor_columns = [
    col
    for col in uci.columns
    if col not in {
        "Result",
        "target",
        "source_dataset",
        "source_row_id",
        "observation_id"
    }
]

print("Predictor columns:", len(uci_predictor_columns))

Predictor columns: 30


In [60]:
uci_feature_strings = (
    uci[uci_predictor_columns]
    .astype(str)
    .agg("|".join, axis=1)
)

uci["feature_hash"] = (
    uci_feature_strings
    .map(
        lambda x: hashlib.sha256(
            x.encode("utf-8")
        ).hexdigest()
    )
)

unique_hashes = sorted(
    uci["feature_hash"].unique()
)

hash_to_group = {
    h: f"uci_group_{i:06d}"
    for i, h in enumerate(
        unique_hashes,
        start=1
    )
}

uci["group_id"] = (
    uci["feature_hash"]
    .map(hash_to_group)
)

In [61]:
uci_group_summary = (
    uci
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        target_classes=("target", "nunique")
    )
    .reset_index()
)

print("Total groups:", len(uci_group_summary))

print(
    "Multi-row groups:",
    (uci_group_summary["rows"] > 1).sum()
)

print(
    "Largest group size:",
    uci_group_summary["rows"].max()
)

print(
    "Mixed-target groups:",
    (uci_group_summary["target_classes"] > 1).sum()
)

Total groups: 5785
Multi-row groups: 64
Largest group size: 2
Mixed-target groups: 64


#### UCI Grouping Result

The cleaned UCI dataset produced 5,785 predictor-based groups from 5,849
observations.

Sixty-four groups contained two observations with identical predictor vectors,
and all 64 of these groups contained conflicting target labels.

These groups are preserved intact rather than removed, split, or manually
resolved. Keeping each predictor-identical group within a single experimental
partition prevents identical feature vectors from leaking across development,
calibration, and test subsets.

No researcher-assigned correction is introduced for the conflicting labels.

### 9.2 Grouped UCI Partitioning

The UCI dataset is partitioned into development, calibration, and test subsets
using the predefined 70/15/15 design.

All observations sharing the same predictor-based `group_id` remain within the
same partition.

Because some predictor-identical groups contain both target classes, partition
assignment considers:

- total observations;
- legitimate observations; and
- phishing observations.

This preserves leakage control while maintaining the target distribution as
closely as possible.

In [62]:
uci_group_table = (
    uci
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        legitimate_rows=("target", lambda s: int((s == 0).sum())),
        phishing_rows=("target", lambda s: int((s == 1).sum()))
    )
    .reset_index()
)

print("Groups:", len(uci_group_table))
print("Rows represented:", uci_group_table["rows"].sum())

print(
    "Mixed-target groups:",
    (
        (uci_group_table["legitimate_rows"] > 0)
        &
        (uci_group_table["phishing_rows"] > 0)
    ).sum()
)

Groups: 5785
Rows represented: 5849
Mixed-target groups: 64


In [63]:
UCI_PARTITION_RATIOS = {
    "development": 0.70,
    "calibration": 0.15,
    "test": 0.15
}

UCI_RANDOM_SEED = 42

rng = np.random.default_rng(UCI_RANDOM_SEED)

uci_groups = uci_group_table.copy()

uci_groups["_random"] = rng.random(
    len(uci_groups)
)

uci_groups = uci_groups.sort_values(
    ["rows", "_random"],
    ascending=[False, True]
)

metrics = [
    "rows",
    "legitimate_rows",
    "phishing_rows"
]

totals = {
    metric: uci_groups[metric].sum()
    for metric in metrics
}

targets = {
    partition: {
        metric: totals[metric] * ratio
        for metric in metrics
    }
    for partition, ratio in UCI_PARTITION_RATIOS.items()
}

assigned = {
    partition: {
        metric: 0
        for metric in metrics
    }
    for partition in UCI_PARTITION_RATIOS
}

uci_partition_assignments = []

for _, row in uci_groups.iterrows():

    partition_scores = {}

    for partition in UCI_PARTITION_RATIOS:

        relative_deficits = []

        for metric in metrics:

            target_value = targets[partition][metric]

            remaining = (
                target_value
                - assigned[partition][metric]
            )

            relative_deficit = (
                remaining
                / max(target_value, 1)
            )

            relative_deficits.append(
                relative_deficit
            )

        partition_scores[partition] = np.mean(
            relative_deficits
        )

    chosen_partition = max(
        partition_scores,
        key=partition_scores.get
    )

    uci_partition_assignments.append({
        "group_id": row["group_id"],
        "partition": chosen_partition
    })

    for metric in metrics:
        assigned[chosen_partition][metric] += row[metric]

In [64]:
uci_partition_map = pd.DataFrame(
    uci_partition_assignments
)

uci = uci.merge(
    uci_partition_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    uci["partition"]
    .value_counts()
)

print()

print(
    uci["partition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

partition
development    4094
calibration     878
test            877
Name: count, dtype: int64

partition
development    69.99
calibration    15.01
test           14.99
Name: proportion, dtype: float64


### 9.3 UCI Partition Integrity Validation

The grouped UCI partitions are validated to confirm that predictor-identical
observations remain within a single partition and that target distributions
remain stable.

Validation confirms that:

- every observation is assigned to exactly one partition;
- no predictor-based `group_id` appears in more than one partition;
- both target classes are represented in all partitions;
- phishing prevalence remains approximately consistent across partitions; and
- all conflicting predictor-identical groups remain intact within a single
  partition.

The test partition is treated as sealed after validation.

In [65]:
print("Total rows:", len(uci))
print(
    "Rows with missing partition:",
    uci["partition"].isna().sum()
)

group_partition_counts = (
    uci
    .groupby("group_id")["partition"]
    .nunique()
)

print(
    "Groups appearing in multiple partitions:",
    (group_partition_counts > 1).sum()
)

partition_target_counts = pd.crosstab(
    uci["partition"],
    uci["target"]
)

print("\nPartition × target counts:")
print(partition_target_counts)

partition_target_prevalence = (
    uci
    .groupby("partition")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(partition_target_prevalence)

Total rows: 5849
Rows with missing partition: 0
Groups appearing in multiple partitions: 0

Partition × target counts:
target          0     1
partition              
calibration   420   458
development  1969  2125
test          441   436

Phishing prevalence (%):
partition
calibration    52.16
development    51.91
test           49.71
Name: target, dtype: float64


In [66]:
uci_mixed_groups = uci_group_table.loc[
    (uci_group_table["legitimate_rows"] > 0)
    &
    (uci_group_table["phishing_rows"] > 0),
    "group_id"
]

mixed_group_partition_counts = (
    uci[
        uci["group_id"].isin(
            uci_mixed_groups
        )
    ]
    .groupby("group_id")["partition"]
    .nunique()
)

print(
    "Mixed-target groups crossing partitions:",
    (mixed_group_partition_counts > 1).sum()
)

Mixed-target groups crossing partitions: 0


In [67]:
assert len(uci) == 5849
assert uci["partition"].isna().sum() == 0
assert (group_partition_counts > 1).sum() == 0
assert (mixed_group_partition_counts > 1).sum() == 0

for partition in [
    "development",
    "calibration",
    "test"
]:
    assert (
        uci.loc[
            uci["partition"] == partition,
            "target"
        ].nunique()
        == 2
    )

print("All UCI partition integrity checks passed.")

All UCI partition integrity checks passed.


## 10. Grouped Cross-Validation for UCI

Five-fold grouped cross-validation is constructed using only the UCI
development partition.

Predictor-based `group_id` values remain intact across folds so that
observations with identical feature vectors cannot appear in both training and
validation portions of a fold.

Because some predictor-identical groups contain conflicting target labels,
fold assignment accounts for:

- total observations;
- legitimate observations; and
- phishing observations.

The calibration and sealed test partitions are excluded from cross-validation.

In [68]:
uci_dev = (
    uci[
        uci["partition"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Development rows:", len(uci_dev))
print("Development groups:", uci_dev["group_id"].nunique())
print(
    "Target distribution:",
    uci_dev["target"].value_counts().to_dict()
)

Development rows: 4094
Development groups: 4050
Target distribution: {1: 2125, 0: 1969}


In [69]:
uci_dev_group_table = (
    uci_dev
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        legitimate_rows=("target", lambda s: int((s == 0).sum())),
        phishing_rows=("target", lambda s: int((s == 1).sum()))
    )
    .reset_index()
)

print(
    "Development groups:",
    len(uci_dev_group_table)
)

print(
    "Rows represented:",
    uci_dev_group_table["rows"].sum()
)

Development groups: 4050
Rows represented: 4094


In [70]:
UCI_N_FOLDS = 5
UCI_CV_RANDOM_SEED = 42

cv_rng = np.random.default_rng(
    UCI_CV_RANDOM_SEED
)

cv_groups = uci_dev_group_table.copy()

cv_groups["_random"] = cv_rng.random(
    len(cv_groups)
)

cv_groups = cv_groups.sort_values(
    ["rows", "_random"],
    ascending=[False, True]
)

metrics = [
    "rows",
    "legitimate_rows",
    "phishing_rows"
]

totals = {
    metric: cv_groups[metric].sum()
    for metric in metrics
}

targets_per_fold = {
    fold: {
        metric: totals[metric] / UCI_N_FOLDS
        for metric in metrics
    }
    for fold in range(1, UCI_N_FOLDS + 1)
}

assigned = {
    fold: {
        metric: 0
        for metric in metrics
    }
    for fold in range(1, UCI_N_FOLDS + 1)
}

uci_fold_assignments = []

for _, row in cv_groups.iterrows():

    fold_scores = {}

    for fold in range(1, UCI_N_FOLDS + 1):

        relative_deficits = []

        for metric in metrics:

            target_value = targets_per_fold[fold][metric]

            remaining = (
                target_value
                - assigned[fold][metric]
            )

            relative_deficit = (
                remaining / max(target_value, 1)
            )

            relative_deficits.append(
                relative_deficit
            )

        fold_scores[fold] = np.mean(
            relative_deficits
        )

    chosen_fold = max(
        fold_scores,
        key=fold_scores.get
    )

    uci_fold_assignments.append({
        "group_id": row["group_id"],
        "cv_fold": chosen_fold
    })

    for metric in metrics:
        assigned[chosen_fold][metric] += row[metric]

In [71]:
uci_fold_map = pd.DataFrame(
    uci_fold_assignments
)

uci_dev = uci_dev.merge(
    uci_fold_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    uci_dev["cv_fold"]
    .value_counts()
    .sort_index()
)

print()

print(
    uci_dev["cv_fold"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

cv_fold
1    820
2    820
3    817
4    818
5    819
Name: count, dtype: int64

cv_fold
1    20.03
2    20.03
3    19.96
4    19.98
5    20.00
Name: proportion, dtype: float64


### 10.1 UCI Cross-Validation Integrity Validation

The five UCI development folds are validated to confirm that predictor-based
leakage-prevention groups remain intact and that the target distribution is
preserved across folds.

Validation confirms that:

- every development observation receives exactly one fold assignment;
- no predictor-based `group_id` appears in more than one fold;
- both target classes are represented in every fold;
- phishing prevalence remains approximately stable across folds; and
- conflicting predictor-identical groups remain intact within a single fold.

In [72]:
print("Development rows:", len(uci_dev))
print(
    "Rows with missing fold:",
    uci_dev["cv_fold"].isna().sum()
)

group_fold_counts = (
    uci_dev
    .groupby("group_id")["cv_fold"]
    .nunique()
)

print(
    "Groups appearing in multiple folds:",
    (group_fold_counts > 1).sum()
)

fold_target_counts = pd.crosstab(
    uci_dev["cv_fold"],
    uci_dev["target"]
)

print("\nFold × target counts:")
print(fold_target_counts)

fold_target_prevalence = (
    uci_dev
    .groupby("cv_fold")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(fold_target_prevalence)

Development rows: 4094
Rows with missing fold: 0
Groups appearing in multiple folds: 0

Fold × target counts:
target     0    1
cv_fold          
1        382  438
2        371  449
3        423  394
4        407  411
5        386  433

Phishing prevalence (%):
cv_fold
1    53.41
2    54.76
3    48.23
4    50.24
5    52.87
Name: target, dtype: float64


In [73]:
dev_uci_mixed_groups = (
    uci_dev_group_table.loc[
        (uci_dev_group_table["legitimate_rows"] > 0)
        &
        (uci_dev_group_table["phishing_rows"] > 0),
        "group_id"
    ]
)

mixed_group_fold_counts = (
    uci_dev[
        uci_dev["group_id"].isin(
            dev_uci_mixed_groups
        )
    ]
    .groupby("group_id")["cv_fold"]
    .nunique()
)

print(
    "Mixed-target groups crossing folds:",
    (mixed_group_fold_counts > 1).sum()
)

Mixed-target groups crossing folds: 0


In [74]:
assert len(uci_dev) == 4094
assert uci_dev["cv_fold"].isna().sum() == 0
assert (group_fold_counts > 1).sum() == 0
assert (mixed_group_fold_counts > 1).sum() == 0

for fold in range(1, 6):
    assert (
        uci_dev.loc[
            uci_dev["cv_fold"] == fold,
            "target"
        ].nunique()
        == 2
    )

print(
    "All UCI grouped cross-validation "
    "integrity checks passed."
)

All UCI grouped cross-validation integrity checks passed.


## 11. ISCX-URL2016 Binary Dataset Construction

The ISCX-URL2016 source provides a broader multiclass dataset as well as a
benign/phishing subset.

For the SE-BRL binary URL experiment, the cleaned
`ISCX-URL2016 Phishing` subset is used as the canonical modeling dataset.

The broader `ISCX-URL2016 All` dataset is not concatenated with this subset
because the audit established that the benign/phishing observations in the
subset are already contained in the broader source.

The native categorical labels are converted deterministically to the
standardized SE-BRL binary target:

- `phishing` = `1`
- `benign` = `0`

No `spam`, `malware`, or `Defacement` observations are relabeled as phishing.

In [75]:
iscx = data["ISCX-URL2016 Phishing"].copy()

iscx["source_dataset"] = "ISCX-URL2016 Phishing"
iscx["source_row_id"] = np.arange(len(iscx))

iscx["observation_id"] = (
    "ISCX::"
    + iscx["source_row_id"].astype(str)
)

iscx["target"] = (
    iscx["URL_Type_obf_Type"]
    .map({
        "benign": 0,
        "phishing": 1
    })
)

print("Rows:", len(iscx))
print("Unique observation IDs:", iscx["observation_id"].nunique())
print("Missing targets:", iscx["target"].isna().sum())

print("\nNative labels:")
print(
    iscx["URL_Type_obf_Type"]
    .value_counts()
)

print("\nStandardized targets:")
print(
    iscx["target"]
    .value_counts()
    .sort_index()
)

Rows: 14823
Unique observation IDs: 14823
Missing targets: 0

Native labels:
URL_Type_obf_Type
benign      7464
phishing    7359
Name: count, dtype: int64

Standardized targets:
target
0    7464
1    7359
Name: count, dtype: int64


### 11.1 Feature-Vector Leakage Grouping

The cleaned ISCX binary dataset does not provide a raw URL or domain identifier.

Leakage prevention therefore uses the complete engineered predictor vector,
excluding the native class label and preprocessing metadata.

Rows with identical predictor vectors receive the same `group_id` so that
identical model inputs cannot be distributed across different experimental
partitions.

Missing structured values are retained during grouping and are not imputed at
this stage. Statistical imputation remains a learned preprocessing operation
to be fitted only on appropriate training data.

In [76]:
iscx_predictor_columns = [
    col
    for col in iscx.columns
    if col not in {
        "URL_Type_obf_Type",
        "target",
        "source_dataset",
        "source_row_id",
        "observation_id"
    }
]

print(
    "Predictor columns:",
    len(iscx_predictor_columns)
)

Predictor columns: 79


In [77]:
iscx_feature_strings = (
    iscx[iscx_predictor_columns]
    .astype("string")
    .fillna("<NA>")
    .agg("|".join, axis=1)
)

iscx["feature_hash"] = (
    iscx_feature_strings
    .map(
        lambda x: hashlib.sha256(
            x.encode("utf-8")
        ).hexdigest()
    )
)

unique_hashes = sorted(
    iscx["feature_hash"].unique()
)

hash_to_group = {
    h: f"iscx_group_{i:06d}"
    for i, h in enumerate(
        unique_hashes,
        start=1
    )
}

iscx["group_id"] = (
    iscx["feature_hash"]
    .map(hash_to_group)
)

In [78]:
iscx_group_summary = (
    iscx
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        target_classes=("target", "nunique")
    )
    .reset_index()
)

print(
    "Total groups:",
    len(iscx_group_summary)
)

print(
    "Multi-row groups:",
    (iscx_group_summary["rows"] > 1).sum()
)

print(
    "Largest group size:",
    iscx_group_summary["rows"].max()
)

print(
    "Mixed-target groups:",
    (iscx_group_summary["target_classes"] > 1).sum()
)

Total groups: 14823
Multi-row groups: 0
Largest group size: 1
Mixed-target groups: 0


#### ISCX Grouping Result

The cleaned ISCX binary dataset produced 14,823 predictor-based groups from
14,823 observations.

No predictor-identical duplicate groups remained after cleaning, and no group
contained conflicting target labels.

The feature-vector grouping strategy is nevertheless retained so that the
partitioning procedure remains explicitly leakage-aware and reproducible.

### 11.2 Grouped ISCX Partitioning

The ISCX binary dataset is partitioned into development, calibration, and test
subsets using the predefined 70/15/15 experimental design.

Although every cleaned observation currently belongs to a unique predictor
group, partition assignment continues to operate on `group_id` so that the
same leakage-control procedure is preserved.

The partitioning procedure aims to maintain both overall row proportions and
the benign/phishing class distribution.

In [79]:
iscx_group_table = (
    iscx
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        benign_rows=("target", lambda s: int((s == 0).sum())),
        phishing_rows=("target", lambda s: int((s == 1).sum()))
    )
    .reset_index()
)

print("Groups:", len(iscx_group_table))
print("Rows represented:", iscx_group_table["rows"].sum())

Groups: 14823
Rows represented: 14823


In [80]:
ISCX_PARTITION_RATIOS = {
    "development": 0.70,
    "calibration": 0.15,
    "test": 0.15
}

ISCX_RANDOM_SEED = 42

rng = np.random.default_rng(ISCX_RANDOM_SEED)

iscx_groups = iscx_group_table.copy()

iscx_groups["_random"] = rng.random(
    len(iscx_groups)
)

iscx_groups = iscx_groups.sort_values(
    ["rows", "_random"],
    ascending=[False, True]
)

metrics = [
    "rows",
    "benign_rows",
    "phishing_rows"
]

totals = {
    metric: iscx_groups[metric].sum()
    for metric in metrics
}

targets = {
    partition: {
        metric: totals[metric] * ratio
        for metric in metrics
    }
    for partition, ratio in ISCX_PARTITION_RATIOS.items()
}

assigned = {
    partition: {
        metric: 0
        for metric in metrics
    }
    for partition in ISCX_PARTITION_RATIOS
}

iscx_partition_assignments = []

for _, row in iscx_groups.iterrows():

    partition_scores = {}

    for partition in ISCX_PARTITION_RATIOS:

        relative_deficits = []

        for metric in metrics:

            target_value = targets[partition][metric]

            remaining = (
                target_value
                - assigned[partition][metric]
            )

            relative_deficit = (
                remaining
                / max(target_value, 1)
            )

            relative_deficits.append(
                relative_deficit
            )

        partition_scores[partition] = np.mean(
            relative_deficits
        )

    chosen_partition = max(
        partition_scores,
        key=partition_scores.get
    )

    iscx_partition_assignments.append({
        "group_id": row["group_id"],
        "partition": chosen_partition
    })

    for metric in metrics:
        assigned[chosen_partition][metric] += row[metric]

In [81]:
iscx_partition_map = pd.DataFrame(
    iscx_partition_assignments
)

iscx = iscx.merge(
    iscx_partition_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    iscx["partition"]
    .value_counts()
)

print()

print(
    iscx["partition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

partition
development    10375
calibration     2224
test            2224
Name: count, dtype: int64

partition
development    69.99
calibration    15.00
test           15.00
Name: proportion, dtype: float64


### 11.3 ISCX Partition Integrity Validation

The grouped ISCX partitions are validated to confirm that all observations are
assigned exactly once, leakage-prevention groups remain intact, and the
benign/phishing distribution is preserved across partitions.

Validation confirms that:

- every observation is assigned to exactly one partition;
- no predictor-based `group_id` appears in more than one partition;
- both target classes are represented in all partitions; and
- phishing prevalence remains approximately stable across partitions.

The test partition is treated as sealed after validation.

In [82]:
print("Total rows:", len(iscx))
print(
    "Rows with missing partition:",
    iscx["partition"].isna().sum()
)

group_partition_counts = (
    iscx
    .groupby("group_id")["partition"]
    .nunique()
)

print(
    "Groups appearing in multiple partitions:",
    (group_partition_counts > 1).sum()
)

partition_target_counts = pd.crosstab(
    iscx["partition"],
    iscx["target"]
)

print("\nPartition × target counts:")
print(partition_target_counts)

partition_target_prevalence = (
    iscx
    .groupby("partition")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(partition_target_prevalence)

Total rows: 14823
Rows with missing partition: 0
Groups appearing in multiple partitions: 0

Partition × target counts:
target          0     1
partition              
calibration  1133  1091
development  5196  5179
test         1135  1089

Phishing prevalence (%):
partition
calibration    49.06
development    49.92
test           48.97
Name: target, dtype: float64


In [83]:
assert len(iscx) == 14823
assert iscx["partition"].isna().sum() == 0
assert (group_partition_counts > 1).sum() == 0

for partition in [
    "development",
    "calibration",
    "test"
]:
    assert (
        iscx.loc[
            iscx["partition"] == partition,
            "target"
        ].nunique()
        == 2
    )

print("All ISCX partition integrity checks passed.")

All ISCX partition integrity checks passed.


#### ISCX Partition Validation Result

The ISCX binary dataset was successfully partitioned into development,
calibration, and test subsets using the intended 70/15/15 design.

All observations were assigned exactly once, no predictor-based group crossed
partition boundaries, and both benign and phishing classes were retained in
all partitions.

The test partition is treated as sealed from this point onward.

## 12. Grouped Cross-Validation for ISCX

Five-fold grouped cross-validation is constructed using only the ISCX
development partition.

Although every current predictor-based group contains one observation, fold
assignment continues to preserve `group_id` integrity and class balance.

The calibration and sealed test partitions are excluded from cross-validation.

In [84]:
iscx_dev = (
    iscx[
        iscx["partition"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Development rows:", len(iscx_dev))
print("Development groups:", iscx_dev["group_id"].nunique())
print(
    "Target distribution:",
    iscx_dev["target"].value_counts().to_dict()
)

Development rows: 10375
Development groups: 10375
Target distribution: {0: 5196, 1: 5179}


In [85]:
iscx_dev_group_table = (
    iscx_dev
    .groupby("group_id")
    .agg(
        rows=("observation_id", "size"),
        benign_rows=("target", lambda s: int((s == 0).sum())),
        phishing_rows=("target", lambda s: int((s == 1).sum()))
    )
    .reset_index()
)

In [86]:
ISCX_N_FOLDS = 5
ISCX_CV_RANDOM_SEED = 42

cv_rng = np.random.default_rng(
    ISCX_CV_RANDOM_SEED
)

cv_groups = iscx_dev_group_table.copy()

cv_groups["_random"] = cv_rng.random(
    len(cv_groups)
)

cv_groups = cv_groups.sort_values(
    ["rows", "_random"],
    ascending=[False, True]
)

metrics = [
    "rows",
    "benign_rows",
    "phishing_rows"
]

totals = {
    metric: cv_groups[metric].sum()
    for metric in metrics
}

targets_per_fold = {
    fold: {
        metric: totals[metric] / ISCX_N_FOLDS
        for metric in metrics
    }
    for fold in range(1, ISCX_N_FOLDS + 1)
}

assigned = {
    fold: {
        metric: 0
        for metric in metrics
    }
    for fold in range(1, ISCX_N_FOLDS + 1)
}

iscx_fold_assignments = []

for _, row in cv_groups.iterrows():

    fold_scores = {}

    for fold in range(1, ISCX_N_FOLDS + 1):

        relative_deficits = []

        for metric in metrics:

            target_value = targets_per_fold[fold][metric]

            remaining = (
                target_value
                - assigned[fold][metric]
            )

            relative_deficit = (
                remaining / max(target_value, 1)
            )

            relative_deficits.append(
                relative_deficit
            )

        fold_scores[fold] = np.mean(
            relative_deficits
        )

    chosen_fold = max(
        fold_scores,
        key=fold_scores.get
    )

    iscx_fold_assignments.append({
        "group_id": row["group_id"],
        "cv_fold": chosen_fold
    })

    for metric in metrics:
        assigned[chosen_fold][metric] += row[metric]

In [87]:
iscx_fold_map = pd.DataFrame(
    iscx_fold_assignments
)

iscx_dev = iscx_dev.merge(
    iscx_fold_map,
    on="group_id",
    how="left",
    validate="many_to_one"
)

print(
    iscx_dev["cv_fold"]
    .value_counts()
    .sort_index()
)

print()

print(
    iscx_dev["cv_fold"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

cv_fold
1    2075
2    2075
3    2075
4    2075
5    2075
Name: count, dtype: int64

cv_fold
1    20.0
2    20.0
3    20.0
4    20.0
5    20.0
Name: proportion, dtype: float64


### 12.1 ISCX Cross-Validation Integrity Validation

The five ISCX development folds are validated to confirm that predictor-based
group integrity and class balance are preserved.

Validation confirms that:

- every development observation receives exactly one fold assignment;
- no predictor-based `group_id` appears in more than one fold;
- both benign and phishing classes are represented in every fold; and
- phishing prevalence remains approximately stable across folds.

In [88]:
print("Development rows:", len(iscx_dev))
print(
    "Rows with missing fold:",
    iscx_dev["cv_fold"].isna().sum()
)

group_fold_counts = (
    iscx_dev
    .groupby("group_id")["cv_fold"]
    .nunique()
)

print(
    "Groups appearing in multiple folds:",
    (group_fold_counts > 1).sum()
)

fold_target_counts = pd.crosstab(
    iscx_dev["cv_fold"],
    iscx_dev["target"]
)

print("\nFold × target counts:")
print(fold_target_counts)

fold_target_prevalence = (
    iscx_dev
    .groupby("cv_fold")["target"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nPhishing prevalence (%):")
print(fold_target_prevalence)

Development rows: 10375
Rows with missing fold: 0
Groups appearing in multiple folds: 0

Fold × target counts:
target      0     1
cv_fold            
1        1013  1062
2        1061  1014
3        1009  1066
4        1043  1032
5        1070  1005

Phishing prevalence (%):
cv_fold
1    51.18
2    48.87
3    51.37
4    49.73
5    48.43
Name: target, dtype: float64


In [89]:
assert len(iscx_dev) == 10375
assert iscx_dev["cv_fold"].isna().sum() == 0
assert (group_fold_counts > 1).sum() == 0

for fold in range(1, 6):
    assert (
        iscx_dev.loc[
            iscx_dev["cv_fold"] == fold,
            "target"
        ].nunique()
        == 2
    )

print(
    "All ISCX grouped cross-validation "
    "integrity checks passed."
)

All ISCX grouped cross-validation integrity checks passed.


#### ISCX Grouped Cross-Validation Result

The ISCX development partition was successfully divided into five equally
sized folds.

No predictor-based `group_id` crossed fold boundaries, and both target classes
were retained in every fold.

These folds are therefore suitable for leakage-safe model development, with
any learned preprocessing transformation fitted only on the training portion
of each fold.

## 13. Leakage-Safe Learned Preprocessing Policy

Partition construction is completed before any learned preprocessing
transformation is fitted.

Operations whose parameters depend on the observed data distribution must not
be fitted globally because doing so would allow information from validation,
calibration, or test observations to influence model development.

The following operations are therefore deferred to the model-development
pipeline:

- statistical missing-value imputation;
- feature scaling or standardization;
- normalization based on observed training statistics;
- variance-based or other data-dependent feature removal;
- vocabulary or text-representation fitting;
- class weighting or resampling procedures; and
- any transformation whose parameters are estimated from data.

During five-fold development cross-validation, these operations must be fitted
only on the training portion of each fold and then applied to that fold's
validation observations.

After model configuration is finalized, preprocessing may be fitted using the
complete development partition before application to the calibration
partition.

The sealed test partition must not influence preprocessing parameters, feature
selection, threshold selection, or model configuration.

In [90]:
preprocessing_policy = pd.DataFrame([
    {
        "operation": "Missing-value imputation",
        "learned_from_data": True,
        "fit_stage": "Training fold / development only",
        "global_fit_allowed": False
    },
    {
        "operation": "Feature scaling",
        "learned_from_data": True,
        "fit_stage": "Training fold / development only",
        "global_fit_allowed": False
    },
    {
        "operation": "Variance-based feature removal",
        "learned_from_data": True,
        "fit_stage": "Training fold / development only",
        "global_fit_allowed": False
    },
    {
        "operation": "Text vocabulary / vectorizer fitting",
        "learned_from_data": True,
        "fit_stage": "Training fold / development only",
        "global_fit_allowed": False
    },
    {
        "operation": "Class weighting / resampling",
        "learned_from_data": True,
        "fit_stage": "Training fold only",
        "global_fit_allowed": False
    },
    {
        "operation": "Target mapping from documented source semantics",
        "learned_from_data": False,
        "fit_stage": "Deterministic preprocessing",
        "global_fit_allowed": True
    },
    {
        "operation": "Stable observation/group identifier construction",
        "learned_from_data": False,
        "fit_stage": "Deterministic preprocessing",
        "global_fit_allowed": True
    }
])

preprocessing_policy

,operation,learned_from_data,fit_stage,global_fit_allowed
0,Missing-value imputation,True,Training fold / development only,False
1,Feature scaling,True,Training fold / development only,False
2,Variance-based feature removal,True,Training fold / development only,False
3,Text vocabulary / vectorizer fitting,True,Training fold / development only,False
4,Class weighting / resampling,True,Training fold only,False
5,Target mapping from documented source semantics,False,Deterministic preprocessing,True
6,Stable observation/group identifier construction,False,Deterministic preprocessing,True


In [91]:
email_cv_map = (
    email_dev[
        ["observation_id", "cv_fold"]
    ]
    .drop_duplicates()
)

email_binary = email_binary.drop(
    columns=["cv_fold"],
    errors="ignore"
)

email_binary = email_binary.merge(
    email_cv_map,
    on="observation_id",
    how="left",
    validate="one_to_one"
)

In [92]:
phiusiil_cv_map = (
    phiusiil_dev[
        ["observation_id", "cv_fold"]
    ]
    .drop_duplicates()
)

phiusiil = phiusiil.drop(
    columns=["cv_fold"],
    errors="ignore"
)

phiusiil = phiusiil.merge(
    phiusiil_cv_map,
    on="observation_id",
    how="left",
    validate="one_to_one"
)

In [93]:
uci_cv_map = (
    uci_dev[
        ["observation_id", "cv_fold"]
    ]
    .drop_duplicates()
)

uci = uci.drop(
    columns=["cv_fold"],
    errors="ignore"
)

uci = uci.merge(
    uci_cv_map,
    on="observation_id",
    how="left",
    validate="one_to_one"
)

In [94]:
iscx_cv_map = (
    iscx_dev[
        ["observation_id", "cv_fold"]
    ]
    .drop_duplicates()
)

iscx = iscx.drop(
    columns=["cv_fold"],
    errors="ignore"
)

iscx = iscx.merge(
    iscx_cv_map,
    on="observation_id",
    how="left",
    validate="one_to_one"
)

In [95]:
for name, df in {
    "Email": email_binary,
    "PhiUSIIL": phiusiil,
    "UCI": uci,
    "ISCX": iscx
}.items():

    dev_missing = df.loc[
        df["partition"] == "development",
        "cv_fold"
    ].isna().sum()

    nondev_with_fold = df.loc[
        df["partition"] != "development",
        "cv_fold"
    ].notna().sum()

    print(
        f"{name}: "
        f"development missing folds = {dev_missing}, "
        f"calibration/test with folds = {nondev_with_fold}"
    )

Email: development missing folds = 0, calibration/test with folds = 0
PhiUSIIL: development missing folds = 0, calibration/test with folds = 0
UCI: development missing folds = 0, calibration/test with folds = 0
ISCX: development missing folds = 0, calibration/test with folds = 0


## 14. Export Preprocessing and Partition Artifacts

The finalized preprocessing outputs preserve:

- standardized binary targets;
- source provenance;
- stable observation identifiers;
- leakage-prevention group identifiers;
- development, calibration, and test partition assignments; and
- five-fold development cross-validation assignments.

No globally fitted statistical transformation is exported from this stage.

The resulting artifacts serve as fixed inputs for downstream SE-BRL feature
extraction and controlled machine-learning experiments.

In [96]:
PREPROCESSED_DATA_DIR = PROCESSED_DIR / "preprocessed"
PREPROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PARTITION_RESULTS_DIR = (
    PREPROCESSING_RESULTS_DIR / "partitions"
)
PARTITION_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Preprocessed data:", PREPROCESSED_DATA_DIR)
print("Partition results:", PARTITION_RESULTS_DIR)

Preprocessed data: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/data/processed/preprocessed
Partition results: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing/partitions


In [97]:
export_datasets = {
    "email_preprocessed.csv": email_binary,
    "phiusiil_preprocessed.csv": phiusiil,
    "uci_phishing_websites_preprocessed.csv": uci,
    "iscx_url2016_preprocessed.csv": iscx
}

for filename, df in export_datasets.items():
    output_path = PREPROCESSED_DATA_DIR / filename
    df.to_csv(output_path, index=False)

    print(
        f"{filename}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )

email_preprocessed.csv: 38,759 rows × 17 columns
phiusiil_preprocessed.csv: 235,795 rows × 64 columns
uci_phishing_websites_preprocessed.csv: 5,849 rows × 39 columns
iscx_url2016_preprocessed.csv: 14,823 rows × 88 columns


In [98]:
partition_manifests = {
    "email_partition_manifest.csv":
        email_binary[
            [
                "observation_id",
                "source_dataset",
                "target",
                "group_id",
                "partition",
                "cv_fold"
            ]
        ],

    "phiusiil_partition_manifest.csv":
        phiusiil[
            [
                "observation_id",
                "source_dataset",
                "target",
                "group_id",
                "partition",
                "cv_fold"
            ]
        ],

    "uci_partition_manifest.csv":
        uci[
            [
                "observation_id",
                "source_dataset",
                "target",
                "group_id",
                "partition",
                "cv_fold"
            ]
        ],

    "iscx_partition_manifest.csv":
        iscx[
            [
                "observation_id",
                "source_dataset",
                "target",
                "group_id",
                "partition",
                "cv_fold"
            ]
        ]
}

for filename, df in partition_manifests.items():
    output_path = PARTITION_RESULTS_DIR / filename
    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing/partitions/email_partition_manifest.csv
Saved: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing/partitions/phiusiil_partition_manifest.csv
Saved: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing/partitions/uci_partition_manifest.csv
Saved: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing/partitions/iscx_partition_manifest.csv


In [99]:
policy_path = (
    PREPROCESSING_RESULTS_DIR
    / "preprocessing_policy.csv"
)

preprocessing_policy.to_csv(
    policy_path,
    index=False
)

print("Saved:", policy_path)

Saved: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/preprocessing/preprocessing_policy.csv


In [100]:
expected_outputs = [
    PREPROCESSED_DATA_DIR / "email_preprocessed.csv",
    PREPROCESSED_DATA_DIR / "phiusiil_preprocessed.csv",
    PREPROCESSED_DATA_DIR / "uci_phishing_websites_preprocessed.csv",
    PREPROCESSED_DATA_DIR / "iscx_url2016_preprocessed.csv",

    PARTITION_RESULTS_DIR / "email_partition_manifest.csv",
    PARTITION_RESULTS_DIR / "phiusiil_partition_manifest.csv",
    PARTITION_RESULTS_DIR / "uci_partition_manifest.csv",
    PARTITION_RESULTS_DIR / "iscx_partition_manifest.csv",

    PREPROCESSING_RESULTS_DIR / "preprocessing_policy.csv"
]

for path in expected_outputs:
    print(
        f"{'OK' if path.exists() else 'MISSING'} "
        f"{path.relative_to(REPO_ROOT)}"
    )

OK experiments/data/processed/preprocessed/email_preprocessed.csv
OK experiments/data/processed/preprocessed/phiusiil_preprocessed.csv
OK experiments/data/processed/preprocessed/uci_phishing_websites_preprocessed.csv
OK experiments/data/processed/preprocessed/iscx_url2016_preprocessed.csv
OK experiments/results/preprocessing/partitions/email_partition_manifest.csv
OK experiments/results/preprocessing/partitions/phiusiil_partition_manifest.csv
OK experiments/results/preprocessing/partitions/uci_partition_manifest.csv
OK experiments/results/preprocessing/partitions/iscx_partition_manifest.csv
OK experiments/results/preprocessing/preprocessing_policy.csv


## 15. Preprocessing Summary and Documentation

### Completed Preprocessing Tasks

The preprocessing stage established fixed, leakage-aware modeling datasets for
the email, webpage, and URL modalities.

The following tasks were completed:

- loaded the cleaned outputs produced by `02_data_cleaning.ipynb`;
- documented native label semantics and source-specific target eligibility;
- constructed source-supported binary phishing targets without manual
  relabeling;
- preserved source provenance and stable observation identifiers;
- created leakage-prevention grouping identifiers;
- generated grouped 70/15/15 development, calibration, and test partitions;
- created grouped five-fold cross-validation assignments within development
  partitions only;
- validated group integrity, class representation, and partition coverage;
- defined a leakage-safe learned preprocessing policy; and
- exported fixed preprocessing and partition artifacts for downstream stages.

### Email Modeling Population

The email binary dataset contains 38,759 observations.

The phishing-positive class is derived from the Nazario phishing corpus, while
the negative class uses only legitimate/ham observations from CEAS-08, Enron,
and SpamAssassin.

Spam-positive observations from CEAS-08, Enron, and SpamAssassin were not
reclassified as phishing.

The final email partitions contain:

- 27,129 development observations;
- 5,815 calibration observations; and
- 5,815 test observations.

Phishing prevalence remains approximately 4.04% across all partitions and
development cross-validation folds.

### PhiUSIIL

PhiUSIIL contains 235,795 observations grouped by normalized domain.

The standardized target uses:

- `1` = phishing
- `0` = legitimate

Fifty-four domain groups contain observations from both target classes. These
groups were preserved intact without manual correction.

The final partition distribution is approximately:

- 70.01% development;
- 15.00% calibration; and
- 14.99% test.

Domain groups do not cross partition or cross-validation fold boundaries.

### UCI Phishing Websites

The cleaned UCI dataset contains 5,849 observations.

Because the source does not provide a raw URL or domain identifier,
leakage-prevention groups were constructed from complete predictor vectors
excluding the native `Result` label.

Sixty-four predictor-identical groups contain conflicting target labels. These
groups were retained intact and were not manually corrected.

The final partitions contain:

- 4,094 development observations;
- 878 calibration observations; and
- 877 test observations.

No predictor-based group crosses a partition or development fold boundary.

### ISCX-URL2016

The benign/phishing subset of ISCX-URL2016 is used as the canonical binary URL
dataset.

The cleaned subset contains 14,823 observations:

- 7,464 benign; and
- 7,359 phishing.

No predictor-identical groups remained after cleaning.

The final partitions contain:

- 10,375 development observations;
- 2,224 calibration observations; and
- 2,224 test observations.

The development partition was divided into five equal folds of 2,075
observations each.

### Leakage-Safe Learned Preprocessing

No globally fitted statistical preprocessing operation was applied during this
stage.

The following operations remain deferred to the model-development pipeline:

- missing-value imputation;
- feature scaling or standardization;
- data-dependent feature removal;
- text vocabulary or vectorizer fitting;
- class weighting or resampling; and
- other transformations whose parameters are estimated from data.

During cross-validation, these transformations must be fitted only on the
training portion of each development fold.

After model configuration is finalized, preprocessing may be fitted using the
complete development partition before being applied to the calibration
partition.

The sealed test partition must remain excluded from preprocessing fitting,
feature-selection decisions, model configuration, and threshold selection.

### Generated Artifacts

Preprocessed modeling datasets:

`experiments/data/processed/preprocessed/`

Partition manifests:

`experiments/results/preprocessing/partitions/`

Preprocessing policy:

`experiments/results/preprocessing/preprocessing_policy.csv`

### Next Stage

The next stage is `04_feature_extraction.ipynb`.

That notebook will derive the modality-aware SE-BRL behavioral representation
while preserving the fixed observation, group, partition, and cross-validation
assignments created during preprocessing.